# RT-IoT2022 - Exploratory Data Analysis

The goal of this notebook is to analyze the RT-IoT2022 dataset. In particular, we want to understand:

- The size of the dataset
- The available features and their data types
- The presence of missing values or duplicates
- The distribution of normal and anomalous traffic

These results guide the choices made in the next notebooks (preprocessing, data split and evaluation metrics).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/raw/RT_IOT2022.csv")
print("Dataset loaded successfully")

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
df.head()
print(df.columns.tolist())
df.info()

In [ ]:
missing_value = df.isnull().sum()
print("Total missing values:", missing_value.sum())
missing_value[missing_value > 0]

The dataset does not contain missing values, so no imputation is needed.

In [ ]:
duplicates = df.duplicated().sum()
print("Number of duplicated rows:", duplicates)

In [ ]:
df = df.drop(columns=["Unnamed: 0"])
print("New dataset shape:", df.shape)

The column "Unnamed: 0" is only a row index, so it is removed. This must be done before looking for duplicates: since the index is different for every row, it would make every row look unique. After removing it, we find 5195 duplicated rows.

They are removed in the preprocessing phase, before splitting the data, so that the same flow cannot appear both in training and in test.

In [ ]:
numeric_columns = df.select_dtypes(include=["number"]).columns
categorical_columns = df.select_dtypes(include=["object"]).columns

print("Number of numeric columns:", len(numeric_columns))
print("Number of categorical columns:", len(categorical_columns))

print("\nCategorical columns:")
print(categorical_columns.tolist())

print(df["Attack_type"].unique())
print("Number of traffic classes:", df["Attack_type"].nunique())

attack_counts = df["Attack_type"].value_counts()
print(attack_counts)

attack_percentages = df["Attack_type"].value_counts(normalize=True) * 100
print(attack_percentages)

The dataset has 83 input features:

- 81 numeric features (flow statistics such as duration, packet counts, bytes and timing)
- 2 categorical features: 'proto' (protocol) and 'service'

The column 'Attack_type' is the target. It contains 12 traffic classes: 3 normal classes and 9 attack classes.

In [ ]:
attack_counts.plot(kind="bar", figsize=(12,6))

plt.title("Distribution of Traffic Classes")
plt.xlabel("Traffic Class")
plt.ylabel("Number of samples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df["Attack_type"].unique()
df["Attack_type"].value_counts()

In [ ]:
normal_classes = ["MQTT_Publish", "Thing_Speak", "Wipro_bulb"]

df["Label"] = df["Attack_type"].apply(
    lambda x: "Normal" if x in normal_classes else "Anomaly"
)
print(df["Label"].value_counts())

label_percentages = df["Label"].value_counts(normalize = True) * 100
print(label_percentages)

For the anomaly detection, the three normal classes are:

- MQTT_Publish
- Thing_Speak
- Wipro_bulb

All other classes are considered anomalous, so the dataset is strongly unbalanced:

- Normal traffic: 12507 samples (about 10%)
- Anomalous traffic: 110610 samples (about 90%)

A single attack, DOS_SYN_Hping, represents about 77% of the whole dataset. Some classes are very small, for example NMAP_FIN_SCAN (28 samples), Metasploit_Brute_Force_SSH (37) and the normal class Wipro_bulb (253).

In [ ]:
label_counts = df["Label"].value_counts()
label_counts.plot(kind="bar", figsize=(6,4))

plt.title("Normal vs Anomaly Traffic")
plt.xlabel("Traffic Type")
plt.ylabel("Number of Samples")
plt.xticks(rotation = 0)
plt.tight_layout()
plt.show()
df[numeric_columns].describe()

In [ ]:
print("Protocol values:")
print(df["proto"].value_counts())

print("\nService values:")
print(df["service"].value_counts())

Most of the traffic uses TCP. For 'service', the value '-' means that the servvice could not be identified, and it is the most frequent value. 

Both columns are categorical, so they will be converted with one-hot encoding in the preprocessing phase

## Conclusions

Up to now, we can clearly see that:
-  the dataset contains 123117 samples, 83 input features and no missing values. 
- Also, after removing the index column, 5195 duplicated rows are found and must be removed before the split. 
- Normal traffic is only about 10% of the data, while the anomalous traffic is about 90% with, in particular, one attack that covers the 77% of the total data.


These observations lead to three choices for the rest of the project:

- **Anomaly detection approach:** the models are trained only on normal traffic and must recognize everything that deviates from it.
- **Evaluation metrics:** since the classes are unbalanced, accuracy alone is misleading. We use Balanced Accuracy as the main metric, together with recall and False Positive Rate.
- **Rare Classes:** classes with very few samples (for example Wipro_bulb) may be harder to model and need to be checked in the results.